# Notebook 04 — Fire Aggregation

**Purpose:** Aggregate fire detections from point level to district × year level, computing fire count, mean FRP, fire duration, onset timing, and peak burning week.

**Input:** `data/processed/fire_with_districts.csv`  
**Output:** `data/processed/fire_stats.csv`

In [6]:
import pandas as pd

# Load district-mapped fire data 
fire = pd.read_csv('../Data/Processed/fire_with_districts.csv', parse_dates=['acq_date'])
print(f"Loaded: {fire.shape}")
print(fire.dtypes)

Loaded: (1179053, 8)
latitude            float64
longitude           float64
acq_date     datetime64[ns]
year                  int64
season               object
state                object
district             object
frp                 float64
dtype: object


In [7]:
# Derive time features needed for aggregation 
fire['year'] = fire['acq_date'].dt.year
fire['doy']  = fire['acq_date'].dt.dayofyear          # day of year (1-366): onset timing
fire['week'] = fire['acq_date'].dt.isocalendar().week  # ISO week: peak burning week


In [8]:
# Aggregate per district × year 

# fire_count : total detections per year  (proxy for burned area)
# mean_frp   : avg Fire Radiative Power MW (proxy for fire intensity)
# fire_days  : distinct burning days       (duration / spread over time)
# onset_doy  : earliest detection DOY     (when burning season starts)
# peak_week  : ISO week with most fires   (peak burning week)
stats = fire.groupby(['state', 'district', 'year']).agg(
    fire_count = ('frp',      'count'),
    mean_frp   = ('frp',      'mean'),
    fire_days  = ('acq_date', pd.Series.nunique),
    onset_doy  = ('doy',      'min'),
    peak_week  = ('week',     lambda x: x.value_counts().idxmax()),
).reset_index()

print(f"Aggregated shape: {stats.shape}")
print(stats.head(8))


Aggregated shape: (386, 8)
     state district  year  fire_count  mean_frp  fire_days  onset_doy  \
0  Haryana   Ambala  2015         464  4.902823         61        124   
1  Haryana   Ambala  2016         541  5.273956         69        103   
2  Haryana   Ambala  2017         627  4.994769         78         99   
3  Haryana   Ambala  2018        1014  4.836075         93         91   
4  Haryana   Ambala  2019        1282  5.475328         81         92   
5  Haryana   Ambala  2020        1109  5.040920         77        114   
6  Haryana   Ambala  2021        1411  4.694869         85         92   
7  Haryana   Ambala  2022         791  4.934640         83         94   

   peak_week  
0         43  
1         42  
2         42  
3         43  
4         43  
5         43  
6         44  
7         43  


In [9]:
# Sanity checks 
print(f"Years covered     : {sorted(stats['year'].unique())}")
print(f"States covered    : {stats['state'].unique().tolist()}")
print(f"Districts covered : {stats['district'].nunique()}")
print(f"\nFire count — min: {stats['fire_count'].min()}, max: {stats['fire_count'].max()}")
print(f"Mean FRP (MW)  — min: {stats['mean_frp'].min():.1f}, max: {stats['mean_frp'].max():.1f}")
print(f"\nTop 5 district-years by fire count:")
print(stats.nlargest(5, 'fire_count')[['state', 'district', 'year', 'fire_count', 'mean_frp']])


Years covered     : [np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023)]
States covered    : ['Haryana', 'Punjab']
Districts covered : 43

Fire count — min: 3, max: 19313
Mean FRP (MW)  — min: 1.8, max: 20.9

Top 5 district-years by fire count:
      state  district  year  fire_count  mean_frp
373  Punjab   Sangrur  2020       19313  7.267989
374  Punjab   Sangrur  2021       18845  7.704735
371  Punjab   Sangrur  2018       16740  6.676802
212  Punjab  Bathinda  2021       15971  8.207051
211  Punjab  Bathinda  2020       15803  7.564178


In [10]:
# Save aggregated statistics 
# Note: saving directly as fire_stats.csv (no rename step needed)
stats.to_csv('../Data/Processed/fire_stats.csv', index=False)
print("Saved: data/processed/fire_stats.csv")
print(f"   Rows    : {len(stats)}")
print(f"   Columns : {stats.columns.tolist()}")


✅ Saved: data/processed/fire_stats.csv
   Rows    : 386
   Columns : ['state', 'district', 'year', 'fire_count', 'mean_frp', 'fire_days', 'onset_doy', 'peak_week']
